# OBC AKLT circuits — load, simulate, decode

Loads the qpy-packed circuits from Faisal's files, simulates each on Aer's `Statevector` backend, and decodes the resulting wavefunction from the 2-qubit-per-spin encoding

$$|0\rangle \to |00\rangle, \quad |1\rangle \to |01\rangle, \quad |2\rangle \to |10\rangle$$

into the spin-1 ($d=3$) basis on $N$ sites. The $|11\rangle$ subspace is out-of-code and should have zero amplitude — we check that explicitly.

In [1]:
import json
import numpy as np
import qiskit
import qiskit.qpy as qpy
from qiskit.quantum_info import Statevector

print('qiskit', qiskit.__version__)

with open('aklt_circuit_names.json') as f:
    names = json.load(f)

with open('aklt_qiskit_circuits.qpy', 'rb') as f:
    circuits = qpy.load(f)

qiskit_circuits = dict(zip(names, circuits))

for n, c in qiskit_circuits.items():
    print(f'  {n:6s}  num_qubits={c.num_qubits}  depth={c.depth()}  ops={dict(c.count_ops())}')
    print(c.draw())

qiskit 2.4.1
  2a      num_qubits=4  depth=4  ops={'unitary': 6}
     ┌──────────┐            ┌──────────┐            
q_0: ┤1         ├────────────┤1         ├────────────
     │  Unitary │┌──────────┐│  Unitary │┌──────────┐
q_1: ┤0         ├┤1         ├┤0         ├┤1         ├
     ├──────────┤│  Unitary │├──────────┤│  Unitary │
q_2: ┤1         ├┤0         ├┤1         ├┤0         ├
     │  Unitary │└──────────┘│  Unitary │└──────────┘
q_3: ┤0         ├────────────┤0         ├────────────
     └──────────┘            └──────────┘            
  2b_1    num_qubits=4  depth=4  ops={'unitary': 6}
     ┌──────────┐            ┌──────────┐            
q_0: ┤1         ├────────────┤1         ├────────────
     │  Unitary │┌──────────┐│  Unitary │┌──────────┐
q_1: ┤0         ├┤1         ├┤0         ├┤1         ├
     ├──────────┤│  Unitary │├──────────┤│  Unitary │
q_2: ┤1         ├┤0         ├┤1         ├┤0         ├
     │  Unitary │└──────────┘│  Unitary │└──────────┘
q_3: ┤0         ├──

## Decoder

Given a `Statevector` on $2N$ qubits, project onto the code subspace (each qubit pair in $\{|00\rangle, |01\rangle, |10\rangle\}$) and return amplitudes indexed by spin-1 strings of length $N$.

**Qubit ordering.** Qiskit's `Statevector` indexes basis states as $|q_{n-1}\dots q_1 q_0\rangle$ (little-endian — qubit 0 is the rightmost bit of the integer index). We assume site $k$ of the spin chain corresponds to qubit pair $(2k, 2k+1)$, with the pair $(q_{2k}, q_{2k+1})$ read as bits $b_0 b_1$ where $b_0$ is the lower-index qubit. If your appendix uses the opposite intra-pair ordering, set `swap_pair=True` below and re-run.

In [2]:
# Encoding: physical spin -> 2-qubit string read as (q_low, q_high)
PAIR_TO_SPIN = {(0, 0): 0, (0, 1): 1, (1, 0): 2}
SPIN_LABEL = {0: '0', 1: '1', 2: '2'}

def decode_statevector(sv: Statevector, N: int, swap_pair: bool = False, tol: float = 1e-10):
    """Decode a 2N-qubit statevector into spin-1 amplitudes on N sites.

    Returns (amps, leakage) where:
      amps: dict mapping spin-string 's_{N-1}...s_1 s_0' -> complex amplitude
      leakage: total probability outside the code subspace (any pair = |11>)
    """
    vec = np.asarray(sv.data)
    assert vec.size == 2 ** (2 * N), f'expected {2**(2*N)} amplitudes, got {vec.size}'
    amps = {}
    leakage = 0.0
    for idx, amp in enumerate(vec):
        if abs(amp) ** 2 < tol and abs(amp) < tol:
            continue
        # qubit bits, little-endian: bit k of idx == value of qubit k
        bits = [(idx >> k) & 1 for k in range(2 * N)]
        spins = []
        in_code = True
        for k in range(N):
            b0, b1 = bits[2 * k], bits[2 * k + 1]
            if swap_pair:
                b0, b1 = b1, b0
            pair = (b0, b1)
            if pair == (1, 1):
                in_code = False
                break
            spins.append(PAIR_TO_SPIN[pair])
        if not in_code:
            leakage += abs(amp) ** 2
            continue
        # write spin string with site N-1 on the left (matches Qiskit big-endian for the *spin* basis)
        s = ''.join(SPIN_LABEL[s] for s in reversed(spins))
        amps[s] = amps.get(s, 0.0 + 0.0j) + complex(amp)
    return amps, leakage


def pretty(amps, top=None, tol=1e-8):
    items = [(s, a) for s, a in amps.items() if abs(a) > tol]
    items.sort(key=lambda sa: -abs(sa[1]))
    if top is not None:
        items = items[:top]
    for s, a in items:
        sign = '+' if a.imag >= 0 else '-'
        print(f'  |{s}>  : {a.real:+.6f} {sign} {abs(a.imag):.6f}i   (|.|={abs(a):.6f})')
    return items

## Simulate and decode all circuits

We use `Statevector.from_instruction` — pure Qiskit, no Aer needed for this. (Aer is installed in the venv if you later want shots-based runs.)

In [3]:
results = {}
for name, circ in qiskit_circuits.items():
    N = circ.num_qubits // 2
    sv = Statevector.from_instruction(circ)
    amps, leakage = decode_statevector(sv, N)
    in_code_prob = sum(abs(a) ** 2 for a in amps.values())
    results[name] = dict(N=N, sv=sv, amps=amps, leakage=leakage, in_code=in_code_prob)
    print(f'{name:6s}  N={N}  in-code prob={in_code_prob:.10f}  leakage={leakage:.2e}  n_nonzero={sum(1 for a in amps.values() if abs(a) > 1e-8)}')

2a      N=2  in-code prob=0.9999999923  leakage=7.70e-09  n_nonzero=9
2b_1    N=2  in-code prob=0.9999999710  leakage=2.90e-08  n_nonzero=9
2b_2    N=2  in-code prob=1.0000000000  leakage=0.00e+00  n_nonzero=2
2c      N=2  in-code prob=1.0000000000  leakage=0.00e+00  n_nonzero=2
3a_1    N=3  in-code prob=0.9999998396  leakage=1.60e-07  n_nonzero=17
3a_2    N=3  in-code prob=0.9999998963  leakage=1.04e-07  n_nonzero=10
3b_1    N=3  in-code prob=0.9999999125  leakage=8.75e-08  n_nonzero=17
3b_2    N=3  in-code prob=0.9999998705  leakage=1.29e-07  n_nonzero=18
4a_1    N=4  in-code prob=0.9999998756  leakage=1.24e-07  n_nonzero=81
4a_2    N=4  in-code prob=0.9999998912  leakage=1.09e-07  n_nonzero=81
4b_1    N=4  in-code prob=0.9999999657  leakage=3.43e-08  n_nonzero=52
4b_2    N=4  in-code prob=0.9999998459  leakage=1.54e-07  n_nonzero=59


In [4]:
# Inspect a specific circuit's decoded amplitudes
name = '2a'
r = results[name]
print(f'Circuit {name}: N={r["N"]} sites, encoded on {2*r["N"]} qubits')
print(f'  in-code probability: {r["in_code"]:.10f}')
print(f'  leakage (|11> pairs): {r["leakage"]:.2e}')
print(f'  nonzero spin-basis amplitudes:')
_ = pretty(r['amps'])

Circuit 2a: N=2 sites, encoded on 4 qubits
  in-code probability: 0.9999999923
  leakage (|11> pairs): 7.70e-09
  nonzero spin-basis amplitudes:
  |12>  : -0.707216 - 0.000000i   (|.|=0.707216)
  |21>  : +0.706998 + 0.000000i   (|.|=0.706998)
  |22>  : +0.000283 + 0.000001i   (|.|=0.000283)
  |11>  : +0.000189 - 0.000203i   (|.|=0.000277)
  |10>  : -0.000013 + 0.000036i   (|.|=0.000039)
  |20>  : -0.000032 + 0.000003i   (|.|=0.000032)
  |01>  : -0.000029 - 0.000004i   (|.|=0.000029)
  |00>  : -0.000012 + 0.000007i   (|.|=0.000014)
  |02>  : +0.000001 + 0.000010i   (|.|=0.000011)


In [5]:
# Dump top amplitudes for every circuit
for name, r in results.items():
    print(f'\n=== {name}  (N={r["N"]},  leakage={r["leakage"]:.1e}) ===')
    pretty(r['amps'], top=12)


=== 2a  (N=2,  leakage=7.7e-09) ===
  |12>  : -0.707216 - 0.000000i   (|.|=0.707216)
  |21>  : +0.706998 + 0.000000i   (|.|=0.706998)
  |22>  : +0.000283 + 0.000001i   (|.|=0.000283)
  |11>  : +0.000189 - 0.000203i   (|.|=0.000277)
  |10>  : -0.000013 + 0.000036i   (|.|=0.000039)
  |20>  : -0.000032 + 0.000003i   (|.|=0.000032)
  |01>  : -0.000029 - 0.000004i   (|.|=0.000029)
  |00>  : -0.000012 + 0.000007i   (|.|=0.000014)
  |02>  : +0.000001 + 0.000010i   (|.|=0.000011)

=== 2b_1  (N=2,  leakage=2.9e-08) ===
  |02>  : -0.894443 + 0.000000i   (|.|=0.894443)
  |11>  : +0.447182 + 0.000000i   (|.|=0.447182)
  |10>  : -0.000038 - 0.000278i   (|.|=0.000280)
  |22>  : +0.000062 - 0.000007i   (|.|=0.000062)
  |21>  : +0.000024 + 0.000045i   (|.|=0.000051)
  |20>  : +0.000039 - 0.000001i   (|.|=0.000039)
  |00>  : +0.000021 - 0.000008i   (|.|=0.000022)
  |01>  : -0.000015 - 0.000000i   (|.|=0.000015)
  |12>  : +0.000008 - 0.000000i   (|.|=0.000008)

=== 2b_2  (N=2,  leakage=0.0e+00) ===
  |

## If amplitudes look wrong: try the other intra-pair ordering

If the spin strings above don't match the appendix, the appendix may pair qubits in the opposite order within each site. Re-run with `swap_pair=True`.

In [6]:
name = '2a'
sv = results[name]['sv']
N = results[name]['N']
amps_swapped, leakage_swapped = decode_statevector(sv, N, swap_pair=True)
print(f'{name} with swap_pair=True:  leakage={leakage_swapped:.2e}')
pretty(amps_swapped)

2a with swap_pair=True:  leakage=7.70e-09
  |21>  : -0.707216 - 0.000000i   (|.|=0.707216)
  |12>  : +0.706998 + 0.000000i   (|.|=0.706998)
  |11>  : +0.000283 + 0.000001i   (|.|=0.000283)
  |22>  : +0.000189 - 0.000203i   (|.|=0.000277)
  |20>  : -0.000013 + 0.000036i   (|.|=0.000039)
  |10>  : -0.000032 + 0.000003i   (|.|=0.000032)
  |02>  : -0.000029 - 0.000004i   (|.|=0.000029)
  |00>  : -0.000012 + 0.000007i   (|.|=0.000014)
  |01>  : +0.000001 + 0.000010i   (|.|=0.000011)


[('21', (-0.7072155211439745-1.3877787807814457e-17j)),
 ('12', (0.7069979055824354+6.938893903907228e-18j)),
 ('11', (0.00028311266862626934+8.916785382924501e-07j)),
 ('22', (0.00018856335300350337-0.00020268125246171507j)),
 ('20', (-1.348512636022055e-05+3.608314700700305e-05j)),
 ('10', (-3.220764266859133e-05+2.7890469258967654e-06j)),
 ('02', (-2.8643592965948983e-05-4.253267739708078e-06j)),
 ('00', (-1.1888307683977084e-05+6.512704965419558e-06j)),
 ('01', (1.3851553067835954e-06+1.0472738569534433e-05j))]

## Transpile to a native gate basis

Decompose every `unitary` into the IBM-style universal basis $\{$`cx, rz, sx, x`$\}$.

**Caveat (Qiskit 2.4.1 bug found while writing this notebook).** Calling `transpile(circ, basis_gates=...)` directly on these circuits produces *wrong* unitaries for several of them — the `UnitarySynthesis` pass mis-decomposes some of the generic 2-qubit gates (e.g. gate 6 of `3a_2` ends up with operator-distance ≈ 0.4 from the original, and the full-circuit fidelity drops to ~0.79). The `TwoQubitBasisDecomposer` class itself works correctly in isolation, so the workaround is to **decompose every 2-qubit unitary manually first**, then call `transpile` only to handle 1-qubit basis translation and any peephole merging. Stay at `optimization_level ≤ 1` afterwards — `level=3` re-invokes the same buggy resynthesis and corrupts the same circuits again.

If you later target a specific backend, change `BASIS` (e.g. `['cz','rz','sx','x']` for Heron, `['ecr','rz','sx','x']` for Eagle). For a non-CX entangling basis you'll also need to construct `TwoQubitBasisDecomposer` with the corresponding gate.

In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit.synthesis import TwoQubitBasisDecomposer
from qiskit.circuit.library import CXGate

BASIS = ['cx', 'rz', 'sx', 'x']
OPT_LEVEL = 1  # 0 or 1 are safe; 3 re-runs the buggy 2q synthesis
_decomposer = TwoQubitBasisDecomposer(CXGate())

def pre_decompose_2q_unitaries(circ):
    """Decompose every `unitary` instruction on 2 qubits using TwoQubitBasisDecomposer
    directly. Works around a transpile() UnitarySynthesis bug in Qiskit 2.4.1."""
    new = QuantumCircuit(circ.num_qubits)
    for instr in circ.data:
        op = instr.operation
        qidxs = [circ.find_bit(q).index for q in instr.qubits]
        if op.name == 'unitary' and len(qidxs) == 2:
            M = np.asarray(op.to_matrix())
            new.compose(_decomposer(M), qubits=qidxs, inplace=True)
        else:
            new.append(op, instr.qubits)
    return new

transpiled = {}
print(f'{"name":<6}  {"nq":>2}  {"depth_in":>8} -> {"depth_out":>9}   {"2q_in":>5} -> {"cx_out":>6}   {"total_in":>8} -> {"total_out":>9}   ops_out')
for name, circ in qiskit_circuits.items():
    pre = pre_decompose_2q_unitaries(circ)
    tcirc = transpile(pre, basis_gates=BASIS, optimization_level=OPT_LEVEL)
    transpiled[name] = tcirc
    ops_in = dict(circ.count_ops())
    ops_out = dict(tcirc.count_ops())
    two_q_in = ops_in.get('unitary', 0)
    two_q_out = ops_out.get('cx', 0)
    print(f'{name:<6}  {circ.num_qubits:>2}  {circ.depth():>8} -> {tcirc.depth():>9}   {two_q_in:>5} -> {two_q_out:>6}   {sum(ops_in.values()):>8} -> {sum(ops_out.values()):>9}   {ops_out}')

name    nq  depth_in -> depth_out   2q_in -> cx_out   total_in -> total_out   ops_out
2a       4         4 ->        77       6 ->     18          6 ->       212   {'rz': 114, 'sx': 80, 'cx': 18}
2b_1     4         4 ->        77       6 ->     18          6 ->       212   {'rz': 114, 'sx': 80, 'cx': 18}
2b_2     4         2 ->        35       3 ->      7          3 ->        87   {'rz': 47, 'sx': 32, 'cx': 7, 'x': 1}
2c       4         2 ->        41       3 ->      8          3 ->        95   {'rz': 50, 'sx': 36, 'cx': 8, 'x': 1}
3a_1     6         4 ->        77      10 ->     29         10 ->       338   {'rz': 182, 'sx': 126, 'cx': 29, 'x': 1}
3a_2     6         4 ->        77      10 ->     28         10 ->       323   {'rz': 173, 'sx': 121, 'cx': 28, 'x': 1}
3b_1     6         4 ->        77      10 ->     29         10 ->       338   {'rz': 182, 'sx': 126, 'cx': 29, 'x': 1}
3b_2     6         4 ->        77      10 ->     28         10 ->       327   {'rz': 176, 'sx': 122, 'cx'

## Verify the transpiled circuits still produce the same state

Two checks per circuit:
1. **Unitary fidelity** between `Operator(orig)` and `Operator(transpiled)` — this is the strict check, catches any synthesis bug regardless of initial state.
2. **In-code probability** of the transpiled circuit applied to $|0\rangle$, and max amplitude deviation in the spin-1 basis.

For a correct unrolling the unitary fidelity should be 1.0 up to floating-point. If you see anything below 0.9999 here, the transpile pipeline has corrupted the circuit and you should not submit it.

In [8]:
from qiskit.quantum_info import Operator

def fidelity_amps(amps_a, amps_b):
    keys = set(amps_a) | set(amps_b)
    return abs(sum(np.conj(amps_a.get(k, 0)) * amps_b.get(k, 0) for k in keys)) ** 2

print(f'{"name":<6}  {"U_fidelity":>12}  {"in-code":>10}  {"<psi|psi_t>":>14}  max|d_amp|')
for name, tcirc in transpiled.items():
    circ_orig = qiskit_circuits[name]
    N = tcirc.num_qubits // 2

    # 1) Full unitary fidelity (the strict check)
    U0 = Operator(circ_orig).data
    U1 = Operator(tcirc).data
    d = 2 ** tcirc.num_qubits
    u_fid = abs((U0.conj() * U1).sum()) ** 2 / d ** 2

    # 2) Decoded amplitudes on the |0...0> initial state
    sv_t = Statevector.from_instruction(tcirc)
    amps_t, _ = decode_statevector(sv_t, N)
    in_code_t = sum(abs(a) ** 2 for a in amps_t.values())
    amps_orig = results[name]['amps']

    # Align global phase before computing max|delta|
    pivot = max(amps_orig, key=lambda k: abs(amps_orig[k]))
    if abs(amps_t.get(pivot, 0)) > 1e-10:
        phase = amps_orig[pivot] / amps_t[pivot]
        phase /= abs(phase)
    else:
        phase = 1.0
    amps_t_aligned = {k: v * phase for k, v in amps_t.items()}
    keys = set(amps_orig) | set(amps_t_aligned)
    max_d = max(abs(amps_orig.get(k, 0) - amps_t_aligned.get(k, 0)) for k in keys)
    psi_fid = fidelity_amps(amps_orig, amps_t)

    flag = '' if u_fid > 0.9999 else '   <-- BAD: transpile corrupted this circuit'
    print(f'{name:<6}  {u_fid:>12.10f}  {in_code_t:>10.8f}  {psi_fid:>14.10f}  {max_d:.2e}{flag}')

name      U_fidelity     in-code     <psi|psi_t>  max|d_amp|
2a      1.0000000000  0.99999999    0.9999999846  2.96e-15
2b_1    1.0000000000  0.99999997    0.9999999419  1.29e-14
2b_2    1.0000000000  1.00000000    1.0000000000  5.64e-16
2c      1.0000000000  1.00000000    1.0000000000  4.44e-15
3a_1    1.0000000000  0.99999984    0.9999996792  2.19e-13
3a_2    1.0000000000  0.99999990    0.9999997926  9.72e-10
3b_1    1.0000000000  0.99999991    0.9999998251  4.16e-14
3b_2    1.0000000000  0.99999987    0.9999997410  8.53e-14
4a_1    1.0000000000  0.99999988    0.9999997513  1.58e-06
4a_2    0.9999999988  0.99999989    0.9999997817  7.98e-06
4b_1    0.9999999995  0.99999997    0.9999999308  5.87e-06
4b_2    0.9999999982  0.99999984    0.9999996897  1.60e-05


## Draw a transpiled circuit

Look at one of the smaller circuits in the native basis so you can eyeball the structure.

In [9]:
name = '2c'  # try '2a', '2b_2', '2c', '3a_2', ...
tcirc = transpiled[name]
print(f'{name}: {tcirc.count_ops()},  depth={tcirc.depth()},  num_qubits={tcirc.num_qubits}')
print(tcirc.draw(fold=120))

2c: OrderedDict([('rz', 50), ('sx', 36), ('cx', 8), ('x', 1)]),  depth=41,  num_qubits=4
global phase: 2.9459
          ┌────┐    ┌────────────┐    ┌────┐    ┌──────────┐              ┌───┐┌─────────────┐   ┌────┐  ┌────────────┐»
q_0: ─────┤ √X ├────┤ Rz(2.6887) ├────┤ √X ├────┤ Rz(-π/2) ├──────────────┤ X ├┤ Rz(-2.4733) ├───┤ √X ├──┤ Rz(4.5206) ├»
      ┌───┴────┴───┐└───┬────┬───┘┌───┴────┴───┐└──┬────┬──┘┌────────────┐└─┬─┘└────┬───┬────┘┌──┴────┴─┐└────────────┘»
q_1: ─┤ Rz(0.9625) ├────┤ √X ├────┤ Rz(5.1447) ├───┤ √X ├───┤ Rz(12.282) ├──■───────┤ X ├─────┤ Rz(π/2) ├──────────────»
     ┌┴────────────┤    ├────┤    ├────────────┤   ├────┤   ├────────────┤┌───┐┌────┴───┴────┐└──┬────┬─┘┌────────────┐»
q_2: ┤ Rz(-1.7782) ├────┤ √X ├────┤ Rz(4.9205) ├───┤ √X ├───┤ Rz(10.595) ├┤ X ├┤ Rz(0.49294) ├───┤ √X ├──┤ Rz(4.4861) ├»
     ├─────────────┤    ├────┤    ├────────────┤   ├────┤   ├────────────┤└─┬─┘└─┬─────────┬─┘   ├────┤  ├───────────┬┘»
q_3: ┤ Rz(0.42342) ├────┤ √X ├────┤ Rz(5.01

## Save transpiled circuits back to qpy

Persist the native-basis circuits to a new qpy file so you can submit them without re-transpiling.

In [10]:
out_path = 'aklt_qiskit_circuits_native.qpy'
with open(out_path, 'wb') as f:
    qpy.dump([transpiled[n] for n in names], f)
print(f'wrote {out_path} with {len(names)} circuits in basis {BASIS}')

wrote aklt_qiskit_circuits_native.qpy with 12 circuits in basis ['cx', 'rz', 'sx', 'x']


# Submit to real hardware (`ibm_fez`, raw counts)

`ibm_fez` is a Heron-family device whose native entangling gate is `cz` (basis: `cz, rz, sx, x`). We re-transpile with `TwoQubitBasisDecomposer(CZGate())` (avoiding the Qiskit 2.4.1 `UnitarySynthesis` bug as before), append a measurement on every qubit, and submit all 12 circuits in one `SamplerV2` job at 1000 shots each, no error mitigation.

**Cost discipline.** The submission cell creates a job and immediately writes the job ID to disk (`aklt_hardware_job.json`). If the kernel dies you can still fetch results from the IBM dashboard or by ID. Don't re-run the submission cell unless you want to spend budget on a fresh job.

In [11]:
# pip install qiskit_ibm_runtime

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# 1. Save your account to disk (only needs to be done once)
# Replace 'MY_IBM_QUANTUM_TOKEN' with your actual API key from the IBM Quantum dashboard
# QiskitRuntimeService.save_account(
#     'MY_IBM_QUANTUM_TOKEN',
#     url='https://auth.quantum-computing.ibm.com/api',
#     token_name='my_token',
#     overwrite=True
# )

In [13]:
# --- Pick backend: ibm_fez (Heron, CZ basis) ---
from qiskit_ibm_runtime import QiskitRuntimeService

BACKEND_NAME = 'ibm_fez'

service = QiskitRuntimeService()
backend = service.backend(BACKEND_NAME)
status = backend.status()
print(f'Selected backend: {backend.name}')
print(f'  num_qubits:   {backend.num_qubits}')
print(f'  basis_gates:  {backend.basis_gates}')
print(f'  operational:  {status.operational}')
print(f'  pending jobs: {status.pending_jobs}')
print(f'  status msg:   {status.status_msg}')

Selected backend: ibm_fez
  num_qubits:   156
  basis_gates:  ['cz', 'id', 'rz', 'sx', 'x']
  operational:  True
  pending jobs: 0
  status msg:   active


In [14]:
# --- Re-transpile to ibm_fez native CZ basis, with all-qubit measurements ---
# Same pre-decompose workaround as before to avoid the UnitarySynthesis bug,
# but now decomposing into a CZ entangling basis (Heron) instead of CX.

from qiskit.synthesis import TwoQubitBasisDecomposer
from qiskit.circuit.library import CZGate

# Heron entangling gate is CZ; build a decomposer that targets it directly
_cz_decomposer = TwoQubitBasisDecomposer(CZGate())

def pre_decompose_2q_unitaries_cz(circ):
    new = QuantumCircuit(circ.num_qubits)
    for instr in circ.data:
        op = instr.operation
        qidxs = [circ.find_bit(q).index for q in instr.qubits]
        if op.name == "unitary" and len(qidxs) == 2:
            M = np.asarray(op.to_matrix())
            new.compose(_cz_decomposer(M), qubits=qidxs, inplace=True)
        else:
            new.append(op, instr.qubits)
    return new

# Note: ibm_fez advertises translation_method="ibm_dynamic_circuits" which lives in
# the separate qiskit-ibm-transpiler package. Override to the built-in "translator"
# since we have no dynamic-circuit features here.
hw_circuits = {}
for name, circ in qiskit_circuits.items():
    pre = pre_decompose_2q_unitaries_cz(circ)
    pre.measure_all()
    tcirc = transpile(
        pre,
        backend=backend,
        optimization_level=1,
        translation_method="translator",
    )
    hw_circuits[name] = tcirc

print(f"{'name':<6}  {'nq':>2}  depth_phys  cz  total   layout (logical->physical)")
for name, tcirc in hw_circuits.items():
    ops = dict(tcirc.count_ops())
    layout = tcirc.layout.initial_index_layout(filter_ancillas=True) if tcirc.layout else None
    print(f"{name:<6}  {tcirc.num_qubits:>2}  {tcirc.depth():>10}  {ops.get('cz', 0):>3}  {sum(ops.values()):>5}   {layout}")


name    nq  depth_phys  cz  total   layout (logical->physical)
2a      156          74   18    187   [0, 1, 2, 3]
2b_1    156          74   18    187   [0, 1, 2, 3]
2b_2    156          32    7     79   [0, 1, 2, 3]
2c      156          38    8     87   [0, 1, 2, 3]
3a_1    156          74   29    296   [0, 1, 2, 3, 4, 5]
3a_2    156          74   28    287   [0, 1, 2, 3, 4, 5]
3b_1    156          74   29    295   [0, 1, 2, 3, 4, 5]
3b_2    156          74   28    282   [0, 1, 2, 3, 4, 5]
4a_1    156         108   62    607   [0, 1, 2, 3, 4, 5, 6, 7]
4a_2    156         108   62    607   [0, 1, 2, 3, 4, 5, 6, 7]
4b_1    156          74   38    391   [0, 1, 2, 3, 4, 5, 6, 7]
4b_2    156         108   60    589   [0, 1, 2, 3, 4, 5, 6, 7]


In [14]:
# # --- Submit to hardware (SamplerV2, 1000 shots, raw counts) ---
# # Submits all 12 circuits in a single job. Writes the job_id and metadata to disk
# # IMMEDIATELY so you can recover even if this kernel dies before results land.

# import json, datetime
# from qiskit_ibm_runtime import SamplerV2

# SHOTS = 1000
# JOB_INFO_PATH = 'aklt_hardware_job.json'

# ordered_names = list(hw_circuits)  # preserve order so we can map results back by index
# pubs = [hw_circuits[n] for n in ordered_names]

# sampler = SamplerV2(mode=backend)
# sampler.options.default_shots = SHOTS
# job = sampler.run(pubs)

# job_info = {
#     'job_id': job.job_id(),
#     'backend': backend.name,
#     'submitted_at': datetime.datetime.now().isoformat(timespec='seconds'),
#     'shots': SHOTS,
#     'circuits': ordered_names,
#     'note': 'OBC AKLT, ibm_fez (Heron, CZ basis), measure_all, no mitigation',
# }
# with open(JOB_INFO_PATH, 'w') as f:
#     json.dump(job_info, f, indent=2)

# print(f'Submitted job_id={job.job_id()} to {backend.name}')
# print(f'Saved job metadata -> {JOB_INFO_PATH}')
# print(f'Track at: https://quantum.ibm.com/jobs/{job.job_id()}')
# print(f'Current status: {job.status()}')

In [23]:
hw_circuits

{'2a': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db5f41d0>,
 '2b_1': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db769250>,
 '2b_2': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db792f90>,
 '2c': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db5f55d0>,
 '3a_1': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db777e90>,
 '3a_2': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db7b2910>,
 '3b_1': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db768ad0>,
 '3b_2': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db59d4d0>,
 '4a_1': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db75df10>,
 '4a_2': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db742cd0>,
 '4b_1': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db793950>,
 '4b_2': <qiskit.circuit.quantumcircuit.QuantumCircuit at 0x198db777490>}

## Fetch results and decode hardware counts into spin-1 probabilities

The fetch cell blocks until the job finishes. Once results land:
- Each circuit returns a `BitArray` over `2N` classical bits.
- Because we transpiled against the backend, the bitstring positions correspond to *physical* qubits, not the original logical ones. We invert `tcirc.layout` to recover the logical-qubit bitstring before applying the spin-1 decoder.

For each circuit we then report:
- **Hardware probability** per spin-1 string (with shot-noise std).
- **Total leakage**: shots where any qubit pair was measured as `|11⟩` (out-of-code; pure error).
- **Classical fidelity** = $\sum_s \sqrt{p_s^{HW} \cdot p_s^{ideal}}$ — Bhattacharyya overlap with the ideal distribution from the simulation cells above. A clean run gives ≈ 1; lower values quantify how much the hardware noise has smeared the distribution.

## Hellinger fidelity vs the noiseless distribution

For two probability distributions $p, q$ over the spin-1 basis,

$$F_H(p, q) \;=\; \Big( \sum_s \sqrt{p_s\, q_s} \Big)^2,
\qquad d_H(p, q) \;=\; \sqrt{1 - \sqrt{F_H(p, q)}}.$$

(This is the Bhattacharyya overlap — same formula as `classical_fidelity` defined in the decode cell — relabeled under its more common name in the quantum-hardware literature.) We use the **noiseless probability distribution from the statevector simulation** ($p^{ideal}_s = |\langle s|\psi\rangle|^2$, computed in the simulate-all cell above) as the reference. Probability mass that lands in the leakage subspace on hardware is *not* renormalized away — it counts against $F_H$, which is the honest comparison.

## Second submission: `optimization_level=3` to cut CZ count

The first run used `optimization_level=1`, which leaves obvious cancellations on the table. Heron's dominant error channel is the CZ gate (~5e-3 per gate), so circuits with 60+ CZs are gate-error-limited. Pre-decomposing every 2-qubit unitary into CZs (same workaround as before) leaves no `UnitaryGate`s in the circuit, so we can safely run `optimization_level=3` — the buggy `UnitarySynthesis` pass has nothing left to mishandle. Outputs are written to timestamped files so the previous opt=1 run on disk is preserved.

In [17]:
# --- Re-transpile at optimization_level=3 to reduce CZ count ---
# Reuses pre_decompose_2q_unitaries_cz from the earlier transpile cell so transpile()
# never sees a UnitaryGate (which would re-trigger the UnitarySynthesis bug).
# We override translation_method='translator' since ibm_fez advertises an IBM plugin
# that is not installed in this venv. seed_transpiler is fixed for reproducibility.

import datetime
from qiskit.quantum_info import Operator

# RUN_TIMESTAMP is pinned to the existing submitted run so re-running this cell
# does NOT create a new timestamp (which would orphan the fetch/decode cells).
# To submit a fresh job, uncomment the datetime.now() line below and run opt3-submit.
RUN_TIMESTAMP = '2026-05-20T21-27-42'
# RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%dT%H-%M-%S')
print(f'RUN_TIMESTAMP = {RUN_TIMESTAMP}')

hw_circuits_opt3 = {}
for name, circ in qiskit_circuits.items():
    pre = pre_decompose_2q_unitaries_cz(circ)
    pre.measure_all()
    tcirc = transpile(
        pre,
        backend=backend,
        optimization_level=3,
        translation_method='translator',
        seed_transpiler=42,
    )
    hw_circuits_opt3[name] = tcirc

# Comparison table vs the opt=1 run
print()
print(f"{'name':<6}  {'cz_opt1':>7} -> {'cz_opt3':>7}  ({'delta':>5})   {'depth_opt1':>10} -> {'depth_opt3':>10}   {'total_opt1':>10} -> {'total_opt3':>10}")
total_cz1 = total_cz3 = 0
for name, tcirc3 in hw_circuits_opt3.items():
    tcirc1 = hw_circuits[name]
    ops1 = dict(tcirc1.count_ops())
    ops3 = dict(tcirc3.count_ops())
    cz1 = ops1.get('cz', 0)
    cz3 = ops3.get('cz', 0)
    total_cz1 += cz1
    total_cz3 += cz3
    d1, d3 = tcirc1.depth(), tcirc3.depth()
    t1, t3 = sum(ops1.values()), sum(ops3.values())
    delta_str = f'{cz3 - cz1:+d}'
    print(f"{name:<6}  {cz1:>7} -> {cz3:>7}  ({delta_str:>5})   {d1:>10} -> {d3:>10}   {t1:>10} -> {t3:>10}")
print(f"{'TOTAL':<6}  {total_cz1:>7} -> {total_cz3:>7}  ({total_cz3-total_cz1:+5d})")

# Sanity check: U_fidelity vs the original (pre-transpile) circuit on N<=3.
# This catches any synthesis regression introduced at opt=3 before we spend hardware budget.
# We compare against qiskit_circuits[name] (the raw input) up to a global phase.
print()
print('U_fidelity sanity check (skip N=4 to keep this fast):')
print(f"{'name':<6}  {'N':>2}  U_fidelity     status")
for name, tcirc3 in hw_circuits_opt3.items():
    N = qiskit_circuits[name].num_qubits // 2
    if N >= 5:
        print(f"{name:<6}  {N:>2}  (skipped, greater than 8-qubit operator)")
        continue
    # Strip the measurements so Operator() can be computed (measurement is non-unitary).
    tcirc3_no_meas = tcirc3.remove_final_measurements(inplace=False)
    # Need to project the 156-qubit transpiled operator down to the layout's logical qubits.
    # Easier: build an Operator on the 2N logical qubits by extracting from the layout-aware circuit.
    # qiskit's Operator(tcirc) handles this if the circuit has no idle measurements but it still
    # spans 156 qubits. Instead, compose a logical-only equivalent: take the transpiled circuit's
    # decompositions but route them back onto 2N qubits via the initial layout.
    try:
        # Build a 2N-qubit circuit from the operations on the layout's active physical qubits.
        from qiskit import QuantumCircuit as _QC
        init_layout = tcirc3.layout.initial_index_layout(filter_ancillas=True)
        phys_to_logical = {phys: log for log, phys in enumerate(init_layout)}
        logical = _QC(2 * N)
        for instr in tcirc3_no_meas.data:
            qargs = [phys_to_logical[tcirc3_no_meas.find_bit(q).index] for q in instr.qubits]
            if any(q is None for q in qargs):
                continue
            logical.append(instr.operation, qargs)
        U_orig = Operator(qiskit_circuits[name]).data
        U_new = Operator(logical).data
        d = 2 ** (2 * N)
        u_fid = abs((U_orig.conj() * U_new).sum()) ** 2 / d ** 2
        status = 'OK' if u_fid > 0.9999 else 'BAD -- DO NOT SUBMIT'
        print(f"{name:<6}  {N:>2}  {u_fid:.10f}  {status}")
    except Exception as e:
        print(f"{name:<6}  {N:>2}  could not verify ({type(e).__name__}: {e})")


RUN_TIMESTAMP = 2026-05-20T21-27-42

name    cz_opt1 -> cz_opt3  (delta)   depth_opt1 -> depth_opt3   total_opt1 -> total_opt3
2a           18 ->      18  (   +0)           74 ->         62          187 ->        157
2b_1         18 ->      18  (   +0)           74 ->         62          187 ->        157
2b_2          7 ->       7  (   +0)           32 ->         28           79 ->         69
2c            8 ->       8  (   +0)           38 ->         33           87 ->         73
3a_1         29 ->      29  (   +0)           74 ->         62          296 ->        247
3a_2         28 ->      28  (   +0)           74 ->         62          287 ->        238
3b_1         29 ->      29  (   +0)           74 ->         62          295 ->        249
3b_2         28 ->      28  (   +0)           74 ->         62          282 ->        237
4a_1         62 ->      62  (   +0)          108 ->         90          607 ->        505
4a_2         62 ->      62  (   +0)          108 ->         90 

In [18]:
# # --- Submit the opt=3 batch to ibm_fez ---
# # Writes a *new* timestamped job metadata file alongside the opt=1 artifacts.
# # Make sure RUN_TIMESTAMP from the opt3-transpile cell is in your kernel.

# import json
# from qiskit_ibm_runtime import SamplerV2

# SHOTS = 1000
# JOB_INFO_PATH_OPT3 = f'aklt_hardware_job_{RUN_TIMESTAMP}.json'

# ordered_names_opt3 = list(hw_circuits_opt3)
# pubs = [hw_circuits_opt3[n] for n in ordered_names_opt3]

# sampler = SamplerV2(mode=backend)
# sampler.options.default_shots = SHOTS
# job_opt3 = sampler.run(pubs)

# job_info_opt3 = {
#     'job_id': job_opt3.job_id(),
#     'backend': backend.name,
#     'submitted_at': datetime.datetime.now().isoformat(timespec='seconds'),
#     'run_timestamp': RUN_TIMESTAMP,
#     'shots': SHOTS,
#     'opt_level': 3,
#     'compared_to': 'aklt_hardware_job.json',
#     'circuits': ordered_names_opt3,
#     'note': 'OBC AKLT, ibm_fez, CZ basis, opt=3, seed_transpiler=42, measure_all, no mitigation',
# }
# with open(JOB_INFO_PATH_OPT3, 'w') as f:
#     json.dump(job_info_opt3, f, indent=2)

# print(f'Submitted opt=3 job_id={job_opt3.job_id()} to {backend.name}')
# print(f'Saved job metadata -> {JOB_INFO_PATH_OPT3}')
# print(f'Track at: https://quantum.ibm.com/jobs/{job_opt3.job_id()}')
# print(f'Current status: {job_opt3.status()}')


In [22]:
# --- Fetch results for the opt=3 job (disk-first; no IBM round-trip if counts file exists) ---
# Same logic as the opt=1 fetch cell. RUN_TIMESTAMP comes from the opt3-transpile cell
# (pinned to the existing run); the counts file aklt_hardware_counts_{RUN_TIMESTAMP}.json
# is read from disk if present, otherwise re-fetched from IBM and written once.

import json, os, glob

# Resolve RUN_TIMESTAMP (re-attach if the kernel forgot it)
try:
    RUN_TIMESTAMP
except NameError:
    candidates = sorted(glob.glob("aklt_hardware_job_*.json"))
    candidates = [c for c in candidates if c != "aklt_hardware_job.json"]
    if not candidates:
        raise RuntimeError("No timestamped opt3 job json found.")
    latest = candidates[-1]
    print(f"Re-attaching to latest timestamped job file: {latest}")
    RUN_TIMESTAMP = latest.replace("aklt_hardware_job_", "").replace(".json", "")

JOB_INFO_PATH_OPT3 = f"aklt_hardware_job_{RUN_TIMESTAMP}.json"
COUNTS_PATH_OPT3   = f"aklt_hardware_counts_{RUN_TIMESTAMP}.json"

with open(JOB_INFO_PATH_OPT3) as f:
    job_info_opt3 = json.load(f)
ordered_names_opt3 = job_info_opt3["circuits"]

if os.path.exists(COUNTS_PATH_OPT3):
    with open(COUNTS_PATH_OPT3) as f:
        raw_counts_opt3 = json.load(f)
    print(f"Loaded raw counts from disk: {COUNTS_PATH_OPT3}  (no IBM call)")
else:
    try:
        job_opt3
    except NameError:
        job_opt3 = service.job(job_info_opt3["job_id"])
        print(f"Reattached to job {job_info_opt3['job_id']} on {job_info_opt3['backend']}")
    print(f"Status: {job_opt3.status()}  (this cell blocks until DONE)")
    result_opt3 = job_opt3.result()
    print(f"Got {len(result_opt3)} PUB results.")
    raw_counts_opt3 = {}
    for name, pub_res in zip(ordered_names_opt3, result_opt3):
        bit_array = pub_res.data.meas
        raw_counts_opt3[name] = bit_array.get_counts()
    with open(COUNTS_PATH_OPT3, "w") as f:
        json.dump(raw_counts_opt3, f, indent=2)
    print(f"Saved raw counts -> {COUNTS_PATH_OPT3}")

for name, c in raw_counts_opt3.items():
    print(f"  {name}: {len(c)} distinct bitstrings, {sum(c.values())} shots")


Loaded raw counts from disk: aklt_hardware_counts_2026-05-20T21-27-42.json  (no IBM call)
  2a: 16 distinct bitstrings, 1000 shots
  2b_1: 16 distinct bitstrings, 1000 shots
  2b_2: 12 distinct bitstrings, 1000 shots
  2c: 11 distinct bitstrings, 1000 shots
  3a_1: 42 distinct bitstrings, 1000 shots
  3a_2: 44 distinct bitstrings, 1000 shots
  3b_1: 41 distinct bitstrings, 1000 shots
  3b_2: 45 distinct bitstrings, 1000 shots
  4a_1: 133 distinct bitstrings, 1000 shots
  4a_2: 120 distinct bitstrings, 1000 shots
  4b_1: 102 distinct bitstrings, 1000 shots
  4b_2: 135 distinct bitstrings, 1000 shots


In [27]:
import json, glob
# Pick the latest counts file from the opt=3 run
latest = sorted(glob.glob('aklt_hardware_counts_2026-05-20T21-27-42.json'))[-1]
with open(latest) as f:
    raw_counts = json.load(f)


In [28]:
# --- Decode hardware bitstrings into spin-1 probabilities ---
# Bitstring convention: Qiskit returns big-endian strings (leftmost char = highest
# classical-bit index). measure_all() writes classical bit k = logical qubit k of the
# circuit *before* transpile, so the bitstring positions are already in LOGICAL order
# regardless of how the routing pass laid the qubits out on the device. We index off
# tcirc.num_clbits (= 2N), NOT tcirc.num_qubits (= full chip size).

from collections import defaultdict
from math import sqrt

def logical_bits_from_bitstring(bitstring_be, n_clbits):
    """Return tuple b such that b[k] = value of logical qubit k."""
    assert len(bitstring_be) == n_clbits, (len(bitstring_be), n_clbits)
    # Convert big-endian string to little-endian list: bits[k] = value of clbit k
    return tuple(int(c) for c in bitstring_be[::-1])

def decode_counts(counts, tcirc, swap_pair=False):
    """Map raw bitstring counts -> (spin1_probs, leakage_prob, n_shots)."""
    n_clbits = tcirc.num_clbits
    N = n_clbits // 2
    n_shots = sum(counts.values())
    spin_counts = defaultdict(int)
    leak_count = 0
    for bitstr, c in counts.items():
        bits = logical_bits_from_bitstring(bitstr, n_clbits)
        spins = []
        ok = True
        for k in range(N):
            b0, b1 = bits[2*k], bits[2*k+1]
            if swap_pair:
                b0, b1 = b1, b0
            if (b0, b1) == (1, 1):
                ok = False; break
            spins.append(PAIR_TO_SPIN[(b0, b1)])
        if not ok:
            leak_count += c
        else:
            s = "".join(SPIN_LABEL[x] for x in reversed(spins))
            spin_counts[s] += c
    spin_probs = {s: c/n_shots for s, c in spin_counts.items()}
    return spin_probs, leak_count/n_shots, n_shots

def classical_fidelity(p_hw, p_ideal):
    keys = set(p_hw) | set(p_ideal)
    return sum(sqrt(p_hw.get(k, 0) * p_ideal.get(k, 0)) for k in keys) ** 2

# Run decode + comparison-to-ideal for all 12 circuits
hw_decoded = {}
print(f"{'name':<6}  shots  leakage   in-code   F_cls(HW,ideal)   top hardware bitstrings (probability)")
for name, counts in raw_counts.items():
    tcirc = hw_circuits[name]
    spin_probs, leak, n = decode_counts(counts, tcirc)
    ideal_amps = results[name]["amps"]
    ideal_probs = {s: abs(a)**2 for s, a in ideal_amps.items()}
    f_cls = classical_fidelity(spin_probs, ideal_probs)
    in_code = 1.0 - leak
    top = sorted(spin_probs.items(), key=lambda kv: -kv[1])[:5]
    top_str = "  ".join(f"|{s}>:{p:.3f}" for s, p in top)
    hw_decoded[name] = dict(spin_probs=spin_probs, leakage=leak, n_shots=n, f_cls=f_cls)
    print(f"{name:<6}  {n:>5}  {leak:>7.3f}  {in_code:>7.3f}   {f_cls:>14.4f}   {top_str}")


name    shots  leakage   in-code   F_cls(HW,ideal)   top hardware bitstrings (probability)
2a       1000    0.081    0.919           0.8561   |12>:0.431  |21>:0.425  |01>:0.017  |10>:0.013  |20>:0.011
2b_1     1000    0.077    0.923           0.8251   |02>:0.662  |11>:0.163  |00>:0.034  |12>:0.017  |22>:0.016
2b_2     1000    0.025    0.975           0.9177   |20>:0.720  |11>:0.198  |01>:0.032  |21>:0.013  |22>:0.007
2c       1000    0.025    0.975           0.9187   |01>:0.477  |10>:0.442  |00>:0.025  |11>:0.011  |20>:0.010
3a_1     1000    0.071    0.929           0.7919   |120>:0.252  |210>:0.243  |201>:0.214  |111>:0.086  |200>:0.017
3a_2     1000    0.086    0.914           0.7985   |102>:0.253  |012>:0.240  |021>:0.235  |111>:0.071  |002>:0.027
3b_1     1000    0.057    0.943           0.8473   |020>:0.468  |101>:0.137  |011>:0.127  |110>:0.116  |000>:0.020
3b_2     1000    0.098    0.902           0.7929   |202>:0.500  |112>:0.116  |121>:0.099  |211>:0.082  |012>:0.014
4a_1     

In [30]:
# --- Hellinger fidelity: hardware probability vector vs noiseless ideal ---
from math import sqrt

def hellinger_fidelity(p, q):
    keys = set(p) | set(q)
    return sum(sqrt(p.get(k, 0.0) * q.get(k, 0.0)) for k in keys) ** 2

def hellinger_distance(p, q):
    return sqrt(max(0.0, 1.0 - sqrt(hellinger_fidelity(p, q))))

hellinger_results = {}
print(f"{'name':<6}  {'N':>2}  {'in-code':>8}  {'F_H':>8}  {'d_H':>8}  {'F_H_in_code':>12}")
for name, info in hw_decoded.items():
    N = qiskit_circuits[name].num_qubits // 2
    p_hw = info['spin_probs']           # leakage NOT renormalized in
    p_ideal = {s: abs(a) ** 2 for s, a in results[name]['amps'].items()}

    f_h = hellinger_fidelity(p_hw, p_ideal)
    d_h = hellinger_distance(p_hw, p_ideal)

    # For context: same metric on the renormalized in-code distribution (factors out leakage)
    in_code_mass = sum(p_hw.values())
    if in_code_mass > 0:
        p_hw_norm = {s: v / in_code_mass for s, v in p_hw.items()}
        f_h_in = hellinger_fidelity(p_hw_norm, p_ideal)
    else:
        f_h_in = 0.0

    hellinger_results[name] = dict(F_H=f_h, d_H=d_h, F_H_in_code=f_h_in)
    print(f"{name:<6}  {N:>2}  {in_code_mass:>8.4f}  {f_h:>8.4f}  {d_h:>8.4f}  {f_h_in:>12.4f}")


name     N   in-code       F_H       d_H   F_H_in_code
2a       2    0.9190    0.8561    0.2734        0.9315
2b_1     2    0.9230    0.8251    0.3028        0.8939
2b_2     2    0.9750    0.9177    0.2051        0.9412
2c       2    0.9750    0.9187    0.2038        0.9422
3a_1     3    0.9290    0.7919    0.3319        0.8524
3a_2     3    0.9140    0.7985    0.3262        0.8736
3b_1     3    0.9430    0.8473    0.2820        0.8985
3b_2     3    0.9020    0.7929    0.3310        0.8790
4a_1     4    0.8420    0.6428    0.4453        0.7634
4a_2     4    0.8780    0.6774    0.4206        0.7715
4b_1     4    0.9140    0.7631    0.3556        0.8349
4b_2     4    0.8310    0.6573    0.4350        0.7910


In [31]:
# --- Decode opt=3 counts and compare side-by-side with opt=1 ---
# Reuses decode_counts, classical_fidelity, hellinger_fidelity, hellinger_distance
# defined in earlier cells. Saves the comparison table to disk for the record.

hw_decoded_opt3 = {}
hellinger_results_opt3 = {}

for name, counts in raw_counts_opt3.items():
    tcirc = hw_circuits_opt3[name]
    spin_probs, leak, n = decode_counts(counts, tcirc)
    p_ideal = {s: abs(a) ** 2 for s, a in results[name]['amps'].items()}
    f_cls = classical_fidelity(spin_probs, p_ideal)
    f_h = hellinger_fidelity(spin_probs, p_ideal)
    d_h = hellinger_distance(spin_probs, p_ideal)
    in_code_mass = sum(spin_probs.values())
    if in_code_mass > 0:
        p_norm = {s: v / in_code_mass for s, v in spin_probs.items()}
        f_h_in = hellinger_fidelity(p_norm, p_ideal)
    else:
        f_h_in = 0.0
    hw_decoded_opt3[name] = dict(spin_probs=spin_probs, leakage=leak, n_shots=n, f_cls=f_cls)
    hellinger_results_opt3[name] = dict(F_H=f_h, d_H=d_h, F_H_in_code=f_h_in)

# Side-by-side comparison
print('Side-by-side: opt=1 vs opt=3 (same 12 circuits, same backend, same shots)')
print(f"{'name':<6}  {'N':>2}  {'cz1':>4} {'cz3':>4}  {'leak1':>6} {'leak3':>6}  {'F_H1':>7} {'F_H3':>7}  {'F_H_in1':>8} {'F_H_in3':>8}")
comparison = {}
for name in ordered_names_opt3:
    N = qiskit_circuits[name].num_qubits // 2
    cz1 = dict(hw_circuits[name].count_ops()).get('cz', 0)
    cz3 = dict(hw_circuits_opt3[name].count_ops()).get('cz', 0)
    leak1 = hw_decoded[name]['leakage']
    leak3 = hw_decoded_opt3[name]['leakage']
    fh1 = hellinger_results[name]['F_H']
    fh3 = hellinger_results_opt3[name]['F_H']
    fhi1 = hellinger_results[name]['F_H_in_code']
    fhi3 = hellinger_results_opt3[name]['F_H_in_code']
    comparison[name] = dict(
        N=N, cz_opt1=cz1, cz_opt3=cz3,
        leakage_opt1=leak1, leakage_opt3=leak3,
        F_H_opt1=fh1, F_H_opt3=fh3,
        F_H_in_code_opt1=fhi1, F_H_in_code_opt3=fhi3,
    )
    print(f"{name:<6}  {N:>2}  {cz1:>4} {cz3:>4}  {leak1:>6.3f} {leak3:>6.3f}  {fh1:>7.4f} {fh3:>7.4f}  {fhi1:>8.4f} {fhi3:>8.4f}")

COMPARISON_PATH = f'aklt_hardware_comparison_{RUN_TIMESTAMP}.json'
with open(COMPARISON_PATH, 'w') as f:
    json.dump({
        'run_timestamp': RUN_TIMESTAMP,
        'backend': backend.name,
        'shots': 1000,
        'opt1_job': 'aklt_hardware_job.json',
        'opt3_job': f'aklt_hardware_job_{RUN_TIMESTAMP}.json',
        'opt3_counts': f'aklt_hardware_counts_{RUN_TIMESTAMP}.json',
        'per_circuit': comparison,
    }, f, indent=2)
print(f'Saved comparison -> {COMPARISON_PATH}')


Side-by-side: opt=1 vs opt=3 (same 12 circuits, same backend, same shots)
name     N   cz1  cz3   leak1  leak3     F_H1    F_H3   F_H_in1  F_H_in3
2a       2    18   18   0.081  0.081   0.8561  0.8561    0.9315   0.9315
2b_1     2    18   18   0.077  0.077   0.8251  0.8251    0.8939   0.8939
2b_2     2     7    7   0.025  0.025   0.9177  0.9177    0.9412   0.9412
2c       2     8    8   0.025  0.025   0.9187  0.9187    0.9422   0.9422
3a_1     3    29   29   0.071  0.071   0.7919  0.7919    0.8524   0.8524
3a_2     3    28   28   0.086  0.086   0.7985  0.7985    0.8736   0.8736
3b_1     3    29   29   0.057  0.057   0.8473  0.8473    0.8985   0.8985
3b_2     3    28   28   0.098  0.098   0.7929  0.7929    0.8790   0.8790
4a_1     4    62   62   0.158  0.158   0.6428  0.6428    0.7634   0.7634
4a_2     4    62   62   0.122  0.122   0.6774  0.6774    0.7715   0.7715
4b_1     4    38   38   0.086  0.086   0.7631  0.7631    0.8349   0.8349
4b_2     4    60   60   0.169  0.169   0.6573  0.6

In [32]:
# --- Comparison: qutrit (paper) vs qubit (this work, opt=3 on ibm_fez) Hellinger ---
# Two qubit columns, each combining value and delta-vs-qutrit in one cell:
#   F_H_qb         = raw Hellinger (out-of-code shots count as zero overlap)
#   F_H_in_code_qb = same but with leakage probability renormalized away.

paper_per_family = {
    "2a": (1, 97.57),
    "2b": (1, 98.36),
    "3a": (4, 94.27),
    "3b": (4, 96.99),
    "4a": (7, 88.68),
    "4b": (7, 80.80),
}

rows = [
    ("2a",   "2a"),
    ("2b_1", "2b"),
    ("2b_2", "2b"),
    ("3a_1", "3a"),
    ("3a_2", "3a"),
    ("3b_1", "3b"),
    ("3b_2", "3b"),
    ("4a_1", "4a"),
    ("4a_2", "4a"),
    ("4b_1", "4b"),
    ("4b_2", "4b"),
]

print()
print("Qutrit (paper) vs qubit (this work, opt=3 on ibm_fez, 1000 shots) Hellinger fidelities")
print("=" * 92)
print(f"{'state':<6}  {'CNOTs_qt':>8}  {'CZ_qb':>5}  {'F_H_qt':>7}  {'F_H_qb (delta)':>17}  {'F_H_in_code_qb (delta)':>24}")
print("-" * 92)
for state_label, family in rows:
    cn_qt, fh_qt_pct = paper_per_family[family]
    cz_qb = dict(hw_circuits_opt3[state_label].count_ops()).get("cz", 0)
    fh_qb_pct    = 100.0 * hellinger_results_opt3[state_label]["F_H"]
    fh_in_qb_pct = 100.0 * hellinger_results_opt3[state_label]["F_H_in_code"]
    delta_raw = fh_qb_pct - fh_qt_pct
    delta_in  = fh_in_qb_pct - fh_qt_pct
    raw_str = f"{fh_qb_pct:5.2f}% ({delta_raw:+6.2f}%)"
    in_str  = f"{fh_in_qb_pct:5.2f}% ({delta_in:+6.2f}%)"
    print(f"{state_label:<6}  {cn_qt:>8}  {cz_qb:>5}  {fh_qt_pct:>6.2f}%  {raw_str:>17}  {in_str:>24}")
print()
print("Notes:")
print("  * CNOTs_qt: paper's entangling-gate count in the qutrit encoding.")
print("  * CZ_qb:    entangling-gate count after Cartan-optimal decomposition into CZs")
print("              (Vidal-Dawson bound: <= 3 CZs per generic 2q unitary; ours saturate it).")
print("  * F_H_qt:   paper's qutrit Hellinger fidelity (one value per family).")
print("  * F_H_qb (delta):         qubit Hellinger vs noiseless ideal; out-of-code shots")
print("                            (|11> on any pair) counted as zero overlap.")
print("                            (delta) = F_H_qb - F_H_qt.")
print("  * F_H_in_code_qb (delta): same as F_H_qb but with leakage probability renormalized")
print("                            away. (delta) = F_H_in_code_qb - F_H_qt.")



Qutrit (paper) vs qubit (this work, opt=3 on ibm_fez, 1000 shots) Hellinger fidelities
state   CNOTs_qt  CZ_qb   F_H_qt     F_H_qb (delta)    F_H_in_code_qb (delta)
--------------------------------------------------------------------------------------------
2a             1     18   97.57%   85.61% (-11.96%)          93.15% ( -4.42%)
2b_1           1     18   98.36%   82.51% (-15.85%)          89.39% ( -8.97%)
2b_2           1      7   98.36%   91.77% ( -6.59%)          94.12% ( -4.24%)
3a_1           4     29   94.27%   79.19% (-15.08%)          85.24% ( -9.03%)
3a_2           4     28   94.27%   79.85% (-14.42%)          87.36% ( -6.91%)
3b_1           4     29   96.99%   84.73% (-12.26%)          89.85% ( -7.14%)
3b_2           4     28   96.99%   79.29% (-17.70%)          87.90% ( -9.09%)
4a_1           7     62   88.68%   64.28% (-24.40%)          76.34% (-12.34%)
4a_2           7     62   88.68%   67.74% (-20.94%)          77.15% (-11.53%)
4b_1           7     38   80.80%   76.3